In [ ]:
import polars as pl

## New Dataset

In [ ]:
markets = pl.read_parquet('data/raw/markets.parquet')

In [ ]:
markets.head()

In [ ]:
def analyse_markets():
    print(markets.shape)

    print(markets.columns)

    print(markets.describe())

    print(markets.head(5))

analyse_markets()


In [ ]:
for c in markets.columns:
    print(markets[c].value_counts())

In [ ]:
markets.null_count() / len(markets) * 100

In [ ]:
import polars as pl
import json

# ── 1. Peek at raw outcome_prices to understand format ──────────────────────
closed = markets.filter(pl.col("closed") == True)
print("=== Raw outcome_prices sample ===")
print(closed["outcome_prices"].head(10))

# ── 2. Parse outcome_prices robustly ────────────────────────────────────────
def parse_resolved_yes(x):
    try:
        # Handle both JSON array strings and plain strings
        if x is None:
            return None
        x = x.strip()
        # Try standard JSON parse first
        parsed = json.loads(x)
        return float(parsed[0]) > 0.5
    except Exception:
        try:
            # Handle single-quoted or malformed strings
            x = x.replace("'", '"')
            parsed = json.loads(x)
            return float(parsed[0]) > 0.5
        except Exception:
            return None

closed = closed.with_columns(
    pl.col("outcome_prices").map_elements(
        parse_resolved_yes,
        return_dtype=pl.Boolean
    ).alias("resolved_yes")
)

print("\n=== Resolution breakdown ===")
print(closed["resolved_yes"].value_counts())
print(f"Null resolutions: {closed['resolved_yes'].null_count()}")

# ── 3. Political market filter ───────────────────────────────────────────────
political_keywords = [
    "election", "president", "congress", "senate", "minister",
    "vote", "democrat", "republican", "trump", "biden",
    "harris", "political", "govern", "parliament",
    "geopolit", "ukraine", "israel", "nato", "war", "sanction"
]

political = closed.filter(
    pl.any_horizontal([
        pl.col("question").str.to_lowercase().str.contains(kw)
        for kw in political_keywords
    ])
)

print("\n=== Political market count ===")
print(f"Closed political markets: {political.shape[0]}")
print(political["resolved_yes"].value_counts())

# ── 4. Date range and volume ─────────────────────────────────────────────────
print("\n=== Date range ===")
print(political.select(["created_at", "end_date"]).describe())

print("\n=== Volume distribution ===")
print(political["volume"].cast(pl.Float64).describe())

# ── 5. Sanity check sample questions ─────────────────────────────────────────
print("\n=== Sample political markets ===")
print(political.select(["question", "resolved_yes", "volume"]).sample(10, seed=42))

In [ ]:
# Filter out obvious sports noise
sports_noise = ["o/u", "spread", "over/under", "moneyline", 
                "vs.", "nfl", "nba", "mlb", "nhl", "epl"]

political_clean = political.filter(
    ~pl.any_horizontal([
        pl.col("question").str.to_lowercase().str.contains(kw)
        for kw in sports_noise
    ])
)

print(f"After sports filter: {political_clean.shape[0]}")
print(political_clean["resolved_yes"].value_counts())
print(political_clean.select(["question", "resolved_yes", "volume"]).sample(10, seed=42))

In [ ]:
from datasets import load_dataset

# Stream just the first batch to check schema and content
dataset = load_dataset(
    "SII-WANGZJ/Polymarket_data",
    data_files="quant.parquet",
    streaming=True,
    split="train"
)

# Grab first 1000 rows to inspect
sample = []
for i, row in enumerate(dataset):
    sample.append(row)
    if i >= 999:
        break

quant_sample = pl.DataFrame(sample)

In [ ]:
import polars as pl
quant_sample = pl.DataFrame(sample)

In [ ]:
quant_sample

In [ ]:
print("=== Schema ===")
print(quant_sample.columns)
print(quant_sample.dtypes)

print("\n=== Sample rows ===")
print(quant_sample.head(5))

print("\n=== Unique markets in sample ===")
print(quant_sample["market_id"].n_unique())

# Check overlap with our political markets
political_ids = set(political_clean["id"].to_list())
quant_ids = set(quant_sample["market_id"].to_list())
print(f"\nOverlap in first 1000 rows: {len(political_ids & quant_ids)}")

In [ ]:
political_keywords = [
    "election", "president", "congress", "senate", "minister",
    "vote", "party", "democrat", "republican", "trump", "biden",
    "harris", "political", "govern", "parliament", "prime minister",
    "geopolit", "ukraine", "israel", "nato", "war", "sanction"
]

political = closed.filter(
    pl.any_horizontal([
        pl.col("question").str.to_lowercase().str.contains(kw)
        for kw in political_keywords
    ])
)
print(f"Closed political markets: {political.shape[0]}")
print(political["resolved_yes"].value_counts())

In [ ]:
print(political.select(["created_at", "end_date"]).describe())
print(political["volume"].cast(pl.Float64).describe())

In [ ]:
print(political.select(["question", "resolved_yes", "volume"]).sample(10))

In [ ]:
events = pl.read_csv('data/raw/polymarket_events.csv')
markets = pl.read_csv('data/raw/polymarket_markets.csv')

# Markets Analysis

In [ ]:
markets.null_count()  / len(markets) * 100

In [ ]:
# How many closed markets do we have?
print(markets["closed"].value_counts())

# What does outcomePrices look like - can we derive resolution?
print(markets.select("outcomePrices").head(10))

# Volume and liquidity fields available
vol_cols = [c for c in markets.columns if "volume" in c.lower() 
            or "liquidity" in c.lower()]
print(vol_cols)

# Check category distribution
print(markets["category"].value_counts() 
      if "category" in markets.columns 
      else "no category column")

# Check event linkage
print(markets["event_id"].n_unique())

# Date range of closed markets
print(markets.filter(pl.col("closed") == True)
      .select(["startDate", "endDate"])
      .describe())

In [ ]:
markets

# Events Analysis

In [ ]:
# Category distribution in events
print(events.columns)
print(events["category"].value_counts().sort("count", descending=True).head(20))

# Join markets to events on event_id
combined = markets.join(events.select(["id", "category", "title"]), 
                        left_on="event_id", 
                        right_on="id", 
                        how="left",
                        suffix="_event")

# How many closed markets have a category after join?
print(combined.filter(pl.col("closed") == True)["category"].null_count())
print(combined.filter(pl.col("closed") == True)["category"].value_counts()
      .sort("count", descending=True).head(20))

# Parse outcomePrices to derive resolution for closed markets
import json
closed = combined.filter(pl.col("closed") == True)

# Check if we can cleanly derive YES/NO resolution
sample_prices = closed["outcomePrices"].head(20).to_list()
for p in sample_prices:
    parsed = json.loads(p)
    yes_price = float(parsed[0])
    print(f"YES: {yes_price:.4f} → {'YES' if yes_price > 0.5 else 'NO'}")

In [ ]:
# Check tags and subcategory population
print(events["subcategory"].value_counts()
      .sort("count", descending=True).head(20))

# Tags - what does it look like?
print(events["tags"].head(10))

# How many events have non-null tags?
print(events["tags"].null_count())
print(events.shape[0])

# Try keyword matching on title as fallback
political_keywords = ["election", "president", "congress", "senate", 
                      "minister", "vote", "party", "political", "govern",
                      "trump", "biden", "democrat", "republican"]

political_markets = combined.filter(
    pl.col("closed") == True
).filter(
    pl.any_horizontal([
        pl.col("question").str.to_lowercase().str.contains(kw)
        for kw in political_keywords
    ])
)
print(f"Political markets via keyword matching: {political_markets.shape[0]}")

In [ ]:
import json

# Parse tags and extract labels
def extract_tag_labels(tags_str):
    try:
        tags = json.loads(tags_str)
        return [t["label"] for t in tags]
    except:
        return []

# Add parsed tags to events
events = events.with_columns(
    pl.col("tags").map_elements(
        extract_tag_labels, 
        return_dtype=pl.List(pl.String)
    ).alias("tag_labels")
)

# Check all unique tag labels
all_tags = (events
    .explode("tag_labels")
    ["tag_labels"]
    .value_counts()
    .sort("count", descending=True))
print(all_tags.head(30))

# Filter to political tags
political_tags = ["Politics", "Global Politics", "US Politics", 
                  "Elections", "Political"]

political_events = events.filter(
    pl.col("tag_labels").list.eval(
        pl.element().is_in(political_tags)
    ).list.any()
)
print(f"Political events via tags: {political_events.shape[0]}")

# Join to closed markets
political_closed = combined.filter(
    pl.col("closed") == True
).join(
    political_events.select("id"),
    left_on="event_id",
    right_on="id",
    how="inner"
)
print(f"Closed political markets: {political_closed.shape[0]}")

In [ ]:
# See ALL tags that might be political
political_related = all_tags.filter(
    pl.col("tag_labels").str.to_lowercase().str.contains(
        "polit|elect|govern|democrat|republican|trump|biden|president|congress|senate|vote|war|ukraine|israel|nato|geopolit"
    )
)
print(political_related)

# Also check what tags the 2437 keyword-matched markets have
# to see if we're missing any political tags
keyword_political_event_ids = combined.filter(
    pl.col("closed") == True
).filter(
    pl.any_horizontal([
        pl.col("question").str.to_lowercase().str.contains(kw)
        for kw in political_keywords
    ])
)["event_id"].unique()

# What tags do these events have?
keyword_events_tags = (events
    .filter(pl.col("id").is_in(keyword_political_event_ids))
    .explode("tag_labels")
    ["tag_labels"]
    .value_counts()
    .sort("count", descending=True))
print(keyword_events_tags.head(30))

In [ ]:
political_tags = [
    # Core political
    "Politics", "Global Politics", "US Politics", "Geopolitics",
    "Elections", "World Elections",
    
    # Trump specific (huge chunk of markets)
    "Trump", "Trump Presidency",
    
    # Geographic/country politics
    "US Elections", "UK Politics", "European Politics",
    "Netherlands", "Dutch Election",
    
    # Other political tags visible in results
    "Mentions", "All",  # skip these — too broad
]

# Refined list — exclude generic tags like "All" and "Mentions"
political_tags_final = [
    "Politics", "Global Politics", "US Politics", "Geopolitics",
    "Elections", "World Elections", "Trump", "Trump Presidency",
    "US Elections", "UK Politics", "European Politics",
    "Dutch Election", "gubernatorial election",
    "republican primaries", "World Elections"
]

political_events_final = events.filter(
    pl.col("tag_labels").list.eval(
        pl.element().is_in(political_tags_final)
    ).list.any()
)
print(f"Political events: {political_events_final.shape[0]}")

# Final closed political markets
political_closed_final = combined.filter(
    pl.col("closed") == True
).join(
    political_events_final.select("id"),
    left_on="event_id",
    right_on="id",
    how="inner"
)
print(f"Closed political markets: {political_closed_final.shape[0]}")

# Also check resolution balance
import json
political_closed_final = political_closed_final.with_columns(
    pl.col("outcomePrices").map_elements(
        lambda x: float(json.loads(x)[0]) > 0.5,
        return_dtype=pl.Boolean
    ).alias("resolved_yes")
)
print(political_closed_final["resolved_yes"].value_counts())

# Date range
print(political_closed_final.select(["startDate", "endDate"]).describe())

In [ ]:
# Check the key feature columns are actually populated
feature_cols = [
    "volume", "volume24hr", "volume1wk", "volume1mo", "volume1yr",
    "liquidity", "spread", "bestBid", "bestAsk",
    "competitive", "startDate", "endDate"
]

print("Null counts for key feature columns:")
print(political_closed_final.select(feature_cols).null_count())

# Check days_active is calculable
print("\nSample volume values:")
print(political_closed_final.select(["volume", "volume24hr", "volume1wk"]).describe())

# Check competitive score
print("\nCompetitive score distribution:")
print(political_closed_final["competitive"].describe())

In [ ]:
# Check bestBid and bestAsk for closed markets
# If they reflect live order book they should be near 0 for closed markets
print(political_closed_final.select(["bestBid", "bestAsk"]).describe())

# If most closed markets have bestBid ~0 and bestAsk ~1
# it means the order book is captured at snapshot time not resolution time
print(political_closed_final
      .select(["bestBid", "bestAsk"])
      .filter(pl.col("bestBid") > 0.1)
      .shape)

In [ ]:
# Quick correlation check
# Do higher volume markets resolve YES more often?
print(political_closed_final
      .group_by("resolved_yes")
      .agg([
          pl.col("volume").median().alias("median_volume"),
          pl.col("liquidity").median().alias("median_liquidity"),
          pl.col("spread").median().alias("median_spread"),
          pl.col("bestBid").median().alias("median_bid")
      ]))